In [ ]:

import warnings
 
warnings.simplefilter(action="ignore", category=FutureWarning)
import pickle
 
import pandas as pd
from sklearn.preprocessing import StandardScaler

# 1. Molecular Features

In [ ]:
# ── Read in data ───────────────────────────────────────────────────────────
# Build the clinical+molecular frame directly from the merged Excel.
SRC = "/Users/batalf/Desktop/LGG_Paper_07_01_2026/Molecular_Subtype_Model/data/Molecular_Cohort_with_Clinical.xlsx"
raw = pd.read_excel(SRC, sheet_name="Molecular+Clinical")
print("raw:", raw.shape)

# infer Event: progression occurred before last overall-survival follow-up
raw["Event"] = raw["overall_survival"] > raw["event_free_survival"]

# select + rename to the pipeline's canonical schema
clinical_features = raw[[
    "SubjectID",
    "Legal Sex",
    "Age at Diagnosis (days)",
    # "Cancer Predispositions",
    "Extent of Tumor Resection",
    "Chemotherapy",
    "Radiotherapy",
    "molecular_subtype",
    "event_free_survival",
    "Event",
    "Cohort",
]].rename(columns={
    "Legal Sex":                 "Sex",
    "Age at Diagnosis (days)":   "Age at Diagnosis",
    # "Cancer Predispositions":    "NF1",
    "Radiotherapy":              "Radiation",
    "molecular_subtype":         "Molecular Subtype",
    "event_free_survival":       "Progression Free Survival",
}).set_index("SubjectID")
# NOTE: this Excel has no Tumor Location column, so the count-encoding /
# standardization of Tumor Location from the clinical pipeline is omitted.

# 1. Input Features
## 1.1 Clinical + Molecular Features (NO ResNet)
X_clinical = clinical_features.copy()
X_clinical = X_clinical.drop(columns=["Progression Free Survival", "Event"])

# apply label encoding
X_clinical = X_clinical.replace(
    to_replace={
        "Sex": {
            "Female": 0,
            "Male": 1,
        },
        "Extent of Tumor Resection": {
            "Not Applicable": 0,
            "Unavailable": 0,
            "Biopsy only": 1,
            "Partial resection": 2,
            "Gross/Near total resection": 3,
        },
        "Chemotherapy": {
            "Yes": 1,
            "No": 0,
            "Not Applicable": 0,
            "Not Reported": 0,
            "Unavailable": 0,
        },
        "Radiation": {
            "Yes": 1,
            "No": 0,
            "Not Applicable": 0,
            "Not Reported": 0,
            "Unavailable": 0,
        },
    }
)

# Clinical NF1 = NF-1 *syndrome* (cancer predisposition). Distinct from the
# molecular NF1 alteration below (which becomes `mol_NF1`).
# X_clinical["NF1"] = X_clinical["NF1"].apply(
#     lambda x: 1 if "Neurofibromatosis, Type 1 (NF-1)" in str(x) else 0
# )

# split subjects into Discovery and Replicate cohorts
discovery_clinical = X_clinical[X_clinical["Cohort"] == "Discovery"].copy()
replicate_clinical = X_clinical[X_clinical["Cohort"] == "Replicate"].copy()

# ── Molecular Subtype encoding — MULTI-HOT (multi-label) ───────────────────
# Each alteration is an independent binary flag, so co-driver tumors
# (KIAA1549-BRAF + RTK, NF1 + FGFR, ...) fire every column they carry and
# CDKN2A/B co-deletions are captured. Columns are prefixed `mol_` to avoid
# collision with the clinical NF1 (NF-1 syndrome) column above.
TOKEN_MAP = {
    "KIAA1549-BRAF": "KIAA1549_BRAF",
    "BRAF V600E":    "BRAF_V600E",
    "NF1-germline":  "NF1",          # -> "NF1_germline" to split germline/somatic
    "NF1-somatic":   "NF1",          # -> "NF1_somatic"
    "FGFR":          "FGFR",
    "RTK":           "RTK",
    "IDH":           "IDH",
    "MYB/MYBL1":     "MYB",
    "other MAPK":    "other_MAPK",
    "BRAF/MAPK":     "other_MAPK",
    "MAPK":          "other_MAPK",
    "CDKN2A/B":      "CDKN2A_B",
    "wildtype":      None,           # reference: all-zeros row, no column
}
ALTERATION_COLUMNS = ["KIAA1549_BRAF", "BRAF_V600E", "NF1", "FGFR",
                      "RTK", "IDH", "MYB", "other_MAPK", "CDKN2A_B"]
MOL_COLS = [f"mol_{c}" for c in ALTERATION_COLUMNS]
HISTOLOGY_PREFIXES = {"LGG", "GNG", "GNT"}   # dropped; not molecular alterations

def multihot_row(subtype):
    row = {f"mol_{c}": 0 for c in ALTERATION_COLUMNS}
    for tok in str(subtype).split(","):
        tok = tok.strip()
        if not tok or tok in HISTOLOGY_PREFIXES:
            continue
        if tok not in TOKEN_MAP:
            print(f"  [WARN] unmapped molecular token {tok!r} in {subtype!r}")
            continue
        col = TOKEN_MAP[tok]
        if col is not None:
            row[f"mol_{col}"] = 1
    return pd.Series(row)

for cohort_df in (discovery_clinical, replicate_clinical):
    cohort_df[MOL_COLS] = cohort_df["Molecular Subtype"].apply(multihot_row)

print("\nDiscovery molecular alteration prevalence:")
print(discovery_clinical[MOL_COLS].sum().sort_values(ascending=False))
print("\nReplicate molecular alteration prevalence:")
print(replicate_clinical[MOL_COLS].sum().sort_values(ascending=False))

# Save the multi-hot spec (replaces the fitted OneHotEncoder)
with open("./molecular_subtype_encoder.pkl", "wb") as f:
    pickle.dump({"token_map": TOKEN_MAP,
                 "alteration_columns": ALTERATION_COLUMNS,
                 "prefix": "mol_",
                 "histology_prefixes": HISTOLOGY_PREFIXES}, f)

# drop working column
discovery_clinical = discovery_clinical.drop(columns=["Molecular Subtype"])
replicate_clinical = replicate_clinical.drop(columns=["Molecular Subtype"])

# apply standardization (Age only — no Tumor Location in this data source)
scaler = StandardScaler()
columns = ["Age at Diagnosis"]
scaler.fit(discovery_clinical[columns])
discovery_clinical[columns] = scaler.transform(discovery_clinical[columns])
replicate_clinical[columns] = scaler.transform(replicate_clinical[columns])
with open("./scaler_clinical.pkl", mode="wb") as file:
    pickle.dump(scaler, file)

# apply normalization
normalizer = 3
discovery_clinical["Extent of Tumor Resection"] = discovery_clinical[
    "Extent of Tumor Resection"
].apply(lambda x: x / normalizer)
replicate_clinical["Extent of Tumor Resection"] = replicate_clinical[
    "Extent of Tumor Resection"
].apply(lambda x: x / normalizer)

# concatenate cohorts
X_clinical_molecular = pd.concat([discovery_clinical, replicate_clinical])

# cache clinical+molecular input features
X_clinical_molecular.to_pickle("X_clinical_molecular.pkl")
print(f"\nFinal shape: {X_clinical_molecular.shape}")
print(f"Columns: {X_clinical_molecular.columns.tolist()}")

# 2. Output Features
# select columns
y = clinical_features[["Progression Free Survival", "Event"]].merge(
    X_clinical_molecular["Cohort"].to_frame(), left_index=True, right_index=True
)
# compute PFS in months
y["Progression Free Survival"] = y["Progression Free Survival"].apply(
    lambda x: int(x) / 30.417
)
# cache output features
y.to_pickle("./y.pkl")
print("y shape:", y.shape)

In [ ]:
X_clinical_molecular